# CTDenoiser — W&B Sweep Notebook

Barebones notebook for hyperparameter search with Weights & Biases.

| Cell | Purpose |
|------|---------|
| **Setup** | Install deps, clone/update repo |
| **Test run** | One synthetic smoke run to confirm train.py works |
| **Test W&B run** | One real run with W&B logging to confirm metric logging works |
| **Sweep** | Bayesian HP search across architectures and hyperparameters |

**Prerequisites:** Run `00_build_cache.ipynb` once to build `ldct_cache.h5`, then copy it to `/content/ldct_cache.h5`.

---
## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = 'https://github.com/tsereda/CTDenoiser.git'
BRANCH    = 'main'
REPO_ROOT = '/content/CTDenoiser'

if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull origin {BRANCH}

%cd {REPO_ROOT}
!pip install -q -e . wandb

In [ ]:
import shutil, os

# Adjust these paths to match your Drive layout
DRIVE_CACHE = '/content/drive/MyDrive/ldct_cache.h5'
LOCAL_CACHE = '/content/ldct_cache.h5'

if not os.path.exists(LOCAL_CACHE) and os.path.exists(DRIVE_CACHE):
    print('Copying cache from Drive to local disk (faster I/O)...')
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    print('Done.')

print('Cache available:', os.path.exists(LOCAL_CACHE))

---
## 1 — Test Run (no W&B)

Verifies train.py works end-to-end before touching W&B.  
Uses synthetic data so no cache is needed.

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model redcnn \
    --epochs 1 \
    --synthetic-len 32
print('Smoke test passed.')

---
## 2 — Test W&B Run

Single run with W&B logging enabled to confirm metrics appear in your project  
before launching a full sweep.  Uses real cache data if available, synthetic otherwise.

In [ ]:
import wandb
wandb.login()

In [ ]:
import os

WANDB_PROJECT = 'ctdenoiser'

cache_flag = f'--h5-cache {LOCAL_CACHE}' if os.path.exists(LOCAL_CACHE) else ''
synthetic_flag = '' if os.path.exists(LOCAL_CACHE) else '--synthetic-len 64'

%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model redcnn \
    --epochs 1 \
    --batch-size 8 \
    --patch-size 64 \
    --lr 1e-4 \
    --wandb-project {WANDB_PROJECT} \
    {cache_flag} \
    {synthetic_flag}

print('Test W&B run complete — check your project at https://wandb.ai')

---
## 3 — Hyperparameter Sweep

Bayesian search over LR, batch size, and model architecture.  
Each run delegates entirely to `train.py` — no training logic lives in this notebook.

> **Fix note:** `train_run()` calls `wandb.init()` to receive the sweep config,
> then immediately calls `run.finish()` before spawning the subprocess.
> This avoids two processes holding the same run open simultaneously
> (which was the cause of the `exit status 1` error in the old notebook).

In [ ]:
import wandb

WANDB_PROJECT = 'ctdenoiser'
SWEEP_COUNT   = 20   # total runs; increase for a thorough search

sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val/psnr', 'goal': 'maximize'},
    'parameters': {
        'model':      {'values': ['redcnn', 'dncnn', 'unet', 'ctformer', 'flowmatching']},
        'lr':         {'values': [1e-5, 1e-4, 1e-3]},
        'batch_size': {'values': [8, 16, 32]},
        'patch_size': {'values': [64, 96]},
        'epochs':     {'value': 1},
    },
}

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT)
print('Sweep ID:', sweep_id)

In [ ]:
import subprocess, sys, os
import wandb

def train_run():
    """Called once per sweep trial by wandb.agent."""
    # Step 1: open run to receive sweep-assigned config.
    run = wandb.init()
    cfg = run.config
    run_id = run.id

    # Step 2: close parent's reference BEFORE spawning subprocess.
    # Leaving the run open while a child process also opens the same run
    # causes wandb to reject the second init (exit status 1).
    run.finish()

    # Step 3: build CLI, resuming the run we just created.
    cmd = [
        sys.executable, '-m', 'ctdenoiser.train',
        '--model',           cfg.model,
        '--lr',              str(cfg.lr),
        '--batch-size',      str(cfg.batch_size),
        '--patch-size',      str(cfg.patch_size),
        '--epochs',          str(cfg.epochs),
        '--wandb-project',   WANDB_PROJECT,
        '--flow-steps-eval', '5',
    ]
    if cfg.model == 'flowmatching':
        cmd += ['--flow-steps', '20']
    if os.path.exists(LOCAL_CACHE):
        cmd += ['--h5-cache', LOCAL_CACHE]

    # WANDB_RUN_ID + WANDB_RESUME=must tells train.py to resume the run
    # we already registered rather than opening a new one.
    env = {
        **os.environ,
        'WANDB_RUN_ID': run_id,
        'WANDB_RESUME': 'must',
    }
    subprocess.run(cmd, env=env, check=True, cwd=REPO_ROOT)


wandb.agent(sweep_id, function=train_run, count=SWEEP_COUNT)

---
## Results

Open [wandb.ai](https://wandb.ai) → project `ctdenoiser` → **Sweeps** tab.

Useful views:
- **Parallel coordinates** — which HP combos dominate on `val/psnr`
- **Scatter** `model` vs `val/psnr` — architecture comparison
- **Importance** — which hyperparameter matters most